# Upload GGUF files to HuggingFace

Uploads:
- `gpt-oss-20b.MXFP4.gguf` — native MXFP4, ~13.8 GB
- `gpt-oss-20b-sft-IQ4_XS-hybrid.gguf` — hybrid MXFP4+IQ4_NL+IQ4_XS, ~11.6 GB

In [ ]:
# ── Configure these ──────────────────────────────────────────────────────────
HF_REPO_ID  = "your-username/gpt-oss-20b-sft-GGUF"  # change to your HF username
HF_TOKEN    = os.environ.get("HF_TOKEN", "")         # set env var or paste token here
PRIVATE     = False                                   # set True to keep repo private
# ─────────────────────────────────────────────────────────────────────────────

import os
from huggingface_hub import HfApi, login

FILES = {
    "gpt-oss-20b.MXFP4.gguf":           "MXFP4 native format (~13.8 GB)",
    "gpt-oss-20b-sft-IQ4_XS-hybrid.gguf": "Hybrid: MXFP4 experts + IQ4_XS attention (~11.6 GB)",
}

def file_size(path):
    if not os.path.exists(path): return "❌ not found"
    return f"{os.path.getsize(path)/1e9:.1f} GB"

print("Files to upload:")
for f, desc in FILES.items():
    print(f"  {f:<45} {file_size(f)}  — {desc}")

if not HF_TOKEN:
    raise ValueError("Set HF_TOKEN environment variable or paste your token into HF_TOKEN above.")

In [ ]:
# Login and create repo if it doesn't exist

login(token=HF_TOKEN)
api = HfApi(token=HF_TOKEN)

try:
    api.repo_info(repo_id=HF_REPO_ID, repo_type="model")
    print(f"✅ Repo already exists: https://huggingface.co/{HF_REPO_ID}")
except Exception:
    api.create_repo(repo_id=HF_REPO_ID, repo_type="model", private=PRIVATE)
    print(f"✅ Created repo: https://huggingface.co/{HF_REPO_ID}")

In [ ]:
# Create a minimal model card describing the quantization

MODEL_CARD = f"""---
base_model: unsloth/gpt-oss-20b
tags:
  - gguf
  - quantized
  - mxfp4
  - iq4_xs
---

# GPT-OSS-20B SFT — GGUF

Fine-tuned GPT-OSS-20B (SFT) exported as GGUF for llama.cpp inference.

## Files

| File | Size | Description |
|---|---|---|
| `gpt-oss-20b.MXFP4.gguf` | ~13.8 GB | Native MXFP4 format — highest quality |
| `gpt-oss-20b-sft-IQ4_XS-hybrid.gguf` | ~11.6 GB | Hybrid: MXFP4 experts + IQ4_XS attention |

## Quantization Strategy

GPT-OSS-20B uses MXFP4 MoE expert weights that **cannot be requantized without quality loss**.
The hybrid GGUF keeps expert weights in their native MXFP4 format:

- `ffn_*_exps` (72 tensors) → `copy` — MXFP4 preserved
- `attn_q/k` and other 2880-column tensors → `IQ4_NL` — avoids 256-block fallback
- All other tensors → `IQ4_XS` + imatrix calibration

## Usage

```bash
llama-cli --model gpt-oss-20b-sft-IQ4_XS-hybrid.gguf --n-gpu-layers 99 -p "Your prompt here"
```
"""

with open("README.md", "w") as f:
    f.write(MODEL_CARD)

api.upload_file(
    path_or_fileobj="README.md",
    path_in_repo="README.md",
    repo_id=HF_REPO_ID,
    repo_type="model",
)
print("✅ Model card uploaded")

In [ ]:
# Upload GGUF files
# huggingface_hub handles chunked upload and resume automatically for large files.

for local_path, description in FILES.items():
    if not os.path.exists(local_path):
        print(f"⚠️  Skipping {local_path} — not found")
        continue

    size_gb = os.path.getsize(local_path) / 1e9
    print(f"\nUploading {local_path} ({size_gb:.1f} GB) ...")

    api.upload_file(
        path_or_fileobj=local_path,
        path_in_repo=local_path,
        repo_id=HF_REPO_ID,
        repo_type="model",
    )
    print(f"✅ {local_path} uploaded")

print(f"\n🎉 All done: https://huggingface.co/{HF_REPO_ID}")

In [ ]:
# Optional: verify uploaded files

from huggingface_hub import list_repo_files

print(f"Files in {HF_REPO_ID}:")
for f in list_repo_files(HF_REPO_ID, repo_type="model", token=HF_TOKEN):
    print(f"  {f}")